Refer to https://github.com/airtlab/violence-detection-tests-on-the-airtlab-dataset, using the idea of transfer learning, we load the pretrained weight, using their preprocessing method and transform binary to 3 classes

# Deep learning for automatic violence detection: tests on the AIRTLab dataset

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/airtlab/violence-detection-tests-on-the-airtlab-dataset/blob/master/notebook/Violence_Detection_on_the_AIRTLAB_Dataset.ipynb)

This notebook contains the source code of the experiments presented in
> P. Sernani, N. Falcionelli, S. Tomassini, P. Contardo and A. F. Dragoni, "Deep Learning for Automatic Violence Detection: Tests on the AIRTLab Dataset," in IEEE Access, vol. 9, pp. 160580-160595, 2021, doi: 10.1109/ACCESS.2021.3131315.

The paper is open access and available here: [https://ieeexplore.ieee.org/document/9627980](https://ieeexplore.ieee.org/document/9627980).

The experiments are **accuracy tests of three different deep neural networks**  based on 3D Convolutional Neural Network (3D CNN) and ConvLSTM architectures. Such models perform a classification on samples of the **AIRTLab dataset**.

The dataset is publicly available in a dedicated GitHub repository:
> <https://github.com/airtlab/A-Dataset-for-Automatic-Violence-Detection-in-Videos>

Specifically, we tested three different models:
1. the first model uses [C3D](https://arxiv.org/abs/1412.0767) a 3D CNN pre-trained on the [Sport-1M dataset](https://cs.stanford.edu/people/karpathy/deepvideo/), as a feature extractor and an SVM a classifier. We use the C3D original weights, without retraining, therefore applying transfer learning. Only the SVM is trained from scratch on the AIRTLab dataset.
2. the second model also uses C3D until the first fully connected layer, adding two fully connected layers to obtain an end-to-end network for classification.We use the C3D original weights, without training again, therefore applying transfer learning. Only the two final fully connected layer are trained from scratch on the AIRTLab dataset.
3. the third model is based on the [ConvLSTM architecture](https://arxiv.org/abs/1506.04214), followed by two fully connected layers, getting an end-to-end network for classification. The entire network is trained from scratch on the AIRTLab dataset.

### Note

The results presented in the paper are computed with a **GPU runtime**. **5 randomized tests for each model** were performed, to generalize the performance of the proposed model. Due to the randomization of the dataset splitting and non-deterministic behaviour of GPU computation, the results can slightly change across different runs.

For more information about non-determism on GPU with TensorFlow check <https://github.com/NVIDIA/framework-determinism>.

In [1]:
is_binary = False

## 1 Preliminary Operations
The following cells:
- **download the C3D weights** from <https://github.com/aslucki/C3D_Sport1M_keras>
- **clone the AIRTLab** data repository into the /datarepo directory;
- define the architecture of the **C3D model**, as described by [Tran et al.](https://arxiv.org/abs/1412.0767);
- define some utility functions to **load the C3D weights**, pre-process the dataset videos to get 16-frames chunks at a resolution of 112 x 112, and extract the features with C3D.

In [ ]:
# downloads C3D weights from https://github.com/aslucki/C3D_Sport1M_keras
# !mkdir weights
# !gdown --id 1rlZ-xTkTMjgWKiQFUedRnHlDgQwx6yTm -O weights/weights.h5

mkdir: cannot create directory ‘weights’: File exists
/bin/bash: line 1: gdown: command not found


In [ ]:
# downloads the AIRTLAB dataset for violence detection
# !mkdir /datarepo
# !git clone https://github.com/airtlab/A-Dataset-for-Automatic-Violence-Detection-in-Videos.git /datarepo

Cloning into '/datarepo'...
remote: Enumerating objects: 376, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 376 (delta 3), reused 11 (delta 3), pack-reused 364 (from 1)
Receiving objects: 100% (376/376), 1.02 GiB | 55.55 MiB/s, done.
Resolving deltas: 100% (3/3), done.
Updating files: 100% (355/355), done.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# import os
# os.listdir('/content/drive/MyDrive/violence-movies/violence-detection-dataset')


Mounted at /content/drive


['non-violence', 'low-level violence', 'high-level violence']

In [2]:
# C3D definition
# from keras.models import Sequential, Model
# from keras.layers import Input, Dense, Dropout, Flatten
# from keras.layers import Conv3D, MaxPooling3D, ZeroPadding3D
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv3D, MaxPooling3D, ZeroPadding3D
from tensorflow.keras.layers import Dense, Dropout, Flatten


def create_C3D_model(summary = False):
    """Creates model object with the sequential API: https://keras.io/models/sequential/

    Parameters
    ----------
    summary : bool
              if True, prints the model summary (default False)

    Returns
    -------
    model : Sequential
            The instantiated model
    """

    model = Sequential()
    input_shape = (16, 112, 112, 3)

    model.add(Conv3D(64, (3, 3, 3), activation='relu',
                     padding='same', name='conv1',
                     input_shape=input_shape))
    model.add(MaxPooling3D(pool_size=(1, 2, 2), strides=(1, 2, 2),
                           padding='valid', name='pool1'))
    # 2nd layer group
    model.add(Conv3D(128, (3, 3, 3), activation='relu',
                     padding='same', name='conv2'))
    model.add(MaxPooling3D(pool_size=(2, 2, 2), strides=(2, 2, 2),
                           padding='valid', name='pool2'))
    # 3rd layer group
    model.add(Conv3D(256, (3, 3, 3), activation='relu',
                     padding='same', name='conv3a'))
    model.add(Conv3D(256, (3, 3, 3), activation='relu',
                     padding='same', name='conv3b'))
    model.add(MaxPooling3D(pool_size=(2, 2, 2), strides=(2, 2, 2),
                           padding='valid', name='pool3'))
    # 4th layer group
    model.add(Conv3D(512, (3, 3, 3), activation='relu',
                     padding='same', name='conv4a'))
    model.add(Conv3D(512, (3, 3, 3), activation='relu',
                     padding='same', name='conv4b'))
    model.add(MaxPooling3D(pool_size=(2, 2, 2), strides=(2, 2, 2),
                           padding='valid', name='pool4'))
    # 5th layer group
    model.add(Conv3D(512, (3, 3, 3), activation='relu',
                     padding='same', name='conv5a'))
    model.add(Conv3D(512, (3, 3, 3), activation='relu',
                     padding='same', name='conv5b'))
    model.add(ZeroPadding3D(padding=((0, 0), (0, 1), (0, 1)), name='zeropad5'))
    model.add(MaxPooling3D(pool_size=(2, 2, 2), strides=(2, 2, 2),
                           padding='valid', name='pool5'))
    model.add(Flatten())
    # FC layers group
    model.add(Dense(4096, activation='relu', name='fc6'))
    model.add(Dropout(.5))
    model.add(Dense(4096, activation='relu', name='fc7'))
    model.add(Dropout(.5))
    model.add(Dense(487, activation='softmax', name='fc8'))

    if summary:
      print(model.summary())

    return model


def create_C3D_refined_model(summary = False):
    """Creates model object with the sequential API: https://keras.io/models/sequential/

    Parameters
    ----------
    summary : bool
              if True, prints the model summary (default False)

    Returns
    -------
    model : Sequential
            The instantiated model
    """

    model = Sequential()
    input_shape = (16, 112, 112, 3)

    model.add(Conv3D(64, (3, 3, 3), activation='relu',
                     padding='same', name='conv1',
                     input_shape=input_shape))
    model.add(MaxPooling3D(pool_size=(1, 2, 2), strides=(1, 2, 2),
                           padding='valid', name='pool1'))
    # 2nd layer group
    model.add(Conv3D(128, (3, 3, 3), activation='relu',
                     padding='same', name='conv2'))
    model.add(MaxPooling3D(pool_size=(2, 2, 2), strides=(2, 2, 2),
                           padding='valid', name='pool2'))
    # 3rd layer group
    model.add(Conv3D(256, (3, 3, 3), activation='relu',
                     padding='same', name='conv3a'))
    model.add(Conv3D(256, (3, 3, 3), activation='relu',
                     padding='same', name='conv3b'))
    model.add(MaxPooling3D(pool_size=(2, 2, 2), strides=(2, 2, 2),
                           padding='valid', name='pool3'))
    # # 4th layer group
    # model.add(Conv3D(512, (3, 3, 3), activation='relu',
    #                  padding='same', name='conv4a'))
    # model.add(Conv3D(512, (3, 3, 3), activation='relu',
    #                  padding='same', name='conv4b'))
    # model.add(MaxPooling3D(pool_size=(2, 2, 2), strides=(2, 2, 2),
    #                        padding='valid', name='pool4'))
    # # 5th layer group
    # model.add(Conv3D(512, (3, 3, 3), activation='relu',
    #                  padding='same', name='conv5a'))
    # model.add(Conv3D(512, (3, 3, 3), activation='relu',
    #                  padding='same', name='conv5b'))
    # model.add(ZeroPadding3D(padding=((0, 0), (0, 1), (0, 1)), name='zeropad5'))
    # model.add(MaxPooling3D(pool_size=(2, 2, 2), strides=(2, 2, 2),
    #                        padding='valid', name='pool5'))
    # model.add(Flatten())
    # # FC layers group
    # model.add(Dense(4096, activation='relu', name='fc6'))
    # model.add(Dropout(.5))
    # model.add(Dense(4096, activation='relu', name='fc7'))
    # model.add(Dropout(.5))
    # model.add(Dense(487, activation='softmax', name='fc8'))

    if summary:
      print(model.summary())

    return model

2026-02-19 23:40:49.888744: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-19 23:40:50.089991: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-19 23:40:50.143547: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-19 23:40:50.161764: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-19 23:40:50.273523: I tensorflow/core/platform/cpu_feature_guar

In [3]:
# Utility functions for the experiments (chunk count, video preprocessing, feature computation, )
# from keras.models import model_from_json, Model
# from tensorflow import keras

from tensorflow import keras
from tensorflow.keras.models import Model

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import csv

def getFeatureExtractor(weightsPath, layer, verbose = False):
    """Gets the C3D feature extractor

    Parameters
    ----------
    weightsPath : str
                  Pathname of the weights file for the C3D model.
    layer : str
            Name of the output layer for the feature extractor
    verbose : bool
              if True print debug logs (default True)

    Returns
    -------

    Model : Model class
            Feature extractor

    """

    # model = create_C3D_model(verbose)

    # # ✅ make the Sequential "called" so model.input exists (Keras 3 requirement)
    # _ = model(np.zeros((1, 16, 112, 112, 3), dtype=np.float32))

    # model.load_weights(weightsPath)
    # model.compile(loss='mean_squared_error', optimizer='sgd')

    # return Model(inputs=model.input,outputs=model.get_layer(layer).output)

    keras.backend.clear_session()

    # base = create_C3D_model(summary=verbose)  # Sequential built with tf.keras layers
    base = create_C3D_refined_model(summary=verbose)  # Sequential built with tf.keras layers

    inp = keras.Input(shape=(16, 112, 112, 3), name="c3d_input")
    x = inp
    pool3_tensor = None

    # Build ONE functional graph using the SAME layer objects
    for l in base.layers:
        x = l(x)
        if l.name == layer:
            pool3_tensor = x

    if pool3_tensor is None:
        raise ValueError(f"Layer '{layer}' not found in model. Available: {[l.name for l in base.layers]}")

    # Load weights AFTER the graph exists
    base.load_weights(weightsPath, by_name=True, skip_mismatch=True)

    feat = keras.Model(inputs=inp, outputs=pool3_tensor, name=f"c3d_{layer}")
    return feat


def count_chunks(videoBasePath):
    """Counts the 16 frames lenght chunks available in a dataset organized in violent and non-violent,
    cam1 and cam2 folders, placed at videoBasePath.

    Parameters
    ----------
    videoBasePath : str
                    Base path of the dataset

    Returns
    -------
    cnt : int
          number of 16 frames lenght chunks in the dataset
    """

    if is_binary == True:
      folders = ['violent', 'non-violent']
    else:
      folders = ['non-violence', 'low-level violence', 'high-level violence']
    cams = ['cam1', 'cam2']
    cnt = 0

    for folder in folders:
        for camName in cams:
            path = os.path.join(videoBasePath, folder, camName)

            videofiles = os.listdir(path)
            for videofile in videofiles:
                filePath = os.path.join(path, videofile)
                video = cv2.VideoCapture(filePath)
                numframes = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
                fps = int(video.get(cv2.CAP_PROP_FPS))
                chunks = numframes//16
                cnt += chunks


    return cnt

def preprocessVideos(videoBasePath, featureBasePath, verbose=True):
    """Preproccess all the videos.

    It extracts samples for the input of C3D from a video dataset, organised in violent and non-violent, cam1 and cam2 folders.
    The samples and the labels are store on two memmap numpy arrays, called samples.mmap and labels.mmap, at "featureBasePath".
    The numpy array with samples has shape (Chunk #, 16, 112, 112, 3), the labels array has shape (Chunk # 16, 112, 112, 3).
    For the AIRTLab dataset the number of chunks is 3537.

    Parameters
    ----------
    videoBasePath : str
                    Pathname to the base of the video repository, which contains two directories,
                    violent and non-violent, which are divided into cam1 and cam2.
    featureBasePath : str
                      it is the pathname of a base where the numpy arrays have to be saved.
    verbose : bool
              if True print debug logs (default True)

    """

    if is_binary == True:
      folders = ['violent', 'non-violent']
    else:
      folders = ['non-violence', 'low-level violence', 'high-level violence']
    cams = ['cam1', 'cam2']
    total_chunks = count_chunks(videoBasePath)
    npSamples = np.memmap(os.path.join(featureBasePath, 'samples.mmap'), dtype=np.float32, mode='w+', shape=(total_chunks, 16, 112, 112, 3))
    npLabels = np.memmap(os.path.join(featureBasePath, 'labels.mmap'), dtype=np.int8, mode='w+', shape=(total_chunks))
    npVideos = np.memmap(os.path.join(featureBasePath, 'videos.mmap'), dtype=np.int8, mode='w+', shape=(total_chunks))
    cnt = 0

    for folder in folders:
        for camName in cams:
            path = os.path.join(videoBasePath, folder, camName)

            videofiles = os.listdir(path)
            for videofile in videofiles:
                filePath = os.path.join(path, videofile)
                video = cv2.VideoCapture(filePath)
                numframes = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
                fps = int(video.get(cv2.CAP_PROP_FPS))
                chunks = numframes//16
                if verbose:
                    print("*** [Video Info] Number of frames: {} - fps: {} - chunks: {}".format(numframes, fps, chunks))
                vid = []
                videoFrames = []
                while True:
                    ret, img = video.read()
                    if not ret:
                        break
                    videoFrames.append(cv2.resize(img, (112, 112)))
                vid = np.array(videoFrames, dtype=np.float32)
                filename = os.path.splitext(videofile)[0]
                chunk_cnt = 0
                for i in range(chunks):
                    X = vid[i*16:i*16+16]
                    chunk_cnt += 1
                    npSamples[cnt] = np.array(X, dtype=np.float32)

                    if is_binary == True:
                      if folder == 'violent':
                        npLabels[cnt] = np.int8(1)
                      else:
                        npLabels[cnt] = np.int8(0)
                    else:
                      if folder == 'high-level violence':
                          npLabels[cnt] = np.int8(2)
                      elif folder == 'low-level violence':
                          npLabels[cnt] = np.int8(1)
                      else:
                          npLabels[cnt] = np.int8(0)
                    npVideos[cnt] = np.int8(filename)

                    cnt += 1

    if verbose:
        print("** Labels **")
        print(npLabels.shape)
        print('\n****\n')
        print("** Samples **")
        print(npSamples.shape)
        print('\n****\n')

    del npSamples
    del npLabels
    del npVideos


In [11]:
# folders to store samples and features during the experiments
!rm -rf airtlabDataset
!mkdir airtlabDataset
!mkdir airtlabDataset/violent
!mkdir airtlabDataset/violent/cam1
!mkdir airtlabDataset/violent/cam2
!mkdir airtlabDataset/non-violent
!mkdir airtlabDataset/non-violent/cam1
!mkdir airtlabDataset/non-violent/cam2
!mkdir airtlabDataset/results

## 4 Video Pre-Processing for End-to-End Networks
The following cell executes the pre-processing on all the videos, transforming them into **16-frames** samples at a resolution of **112 x 112**. The samples (and their labels) are stored into two [memmaps](https://numpy.org/doc/stable/reference/generated/numpy.memmap.html) at the "airtlabDataset" path, "**samples.mmap**" and **"labels.mmap**", to prevent the loading of **all the samples in memory** at the same time.



In [5]:
from pathlib import Path
try:
    BASE_DIR = Path(__file__).resolve().parent.parent
except NameError:
    # Jupyter notebook fallback
    BASE_DIR = Path.cwd().parent

In [ ]:
# preprocessVideos('/datarepo/violence-detection-dataset', 'airtlabDataset', True)
preprocessVideos(BASE_DIR/'data/processed/violence-detection-dataset', 'airtlabDataset', True)


*** [Video Info] Number of frames: 158 - fps: 30 - chunks: 9
*** [Video Info] Number of frames: 159 - fps: 30 - chunks: 9
*** [Video Info] Number of frames: 159 - fps: 30 - chunks: 9
*** [Video Info] Number of frames: 128 - fps: 30 - chunks: 8
*** [Video Info] Number of frames: 261 - fps: 30 - chunks: 16
*** [Video Info] Number of frames: 150 - fps: 30 - chunks: 9
*** [Video Info] Number of frames: 369 - fps: 30 - chunks: 23
*** [Video Info] Number of frames: 159 - fps: 30 - chunks: 9
*** [Video Info] Number of frames: 231 - fps: 30 - chunks: 14
*** [Video Info] Number of frames: 192 - fps: 30 - chunks: 12
*** [Video Info] Number of frames: 74 - fps: 30 - chunks: 4
*** [Video Info] Number of frames: 138 - fps: 30 - chunks: 8
*** [Video Info] Number of frames: 101 - fps: 30 - chunks: 6
*** [Video Info] Number of frames: 153 - fps: 30 - chunks: 9
*** [Video Info] Number of frames: 136 - fps: 30 - chunks: 8
*** [Video Info] Number of frames: 177 - fps: 30 - chunks: 11
*** [Video Info] Num

## 5 Testing End-to-End Networks
The following cells
- define **two end-to-end models** and the code to run the experiments on such models; as with C3D + SVM, the experiments are tests repeated **5 times** with the **stratified shuffle split** cross-validation scheme. In each split 80% of data are used for training, and 20% of data are used for testing. 12,5% of the training data (i.e. 10% of the entire dataset) is used for validation. In other words, in each test **70%** of data are actually for **training**, **10%** for **validation**, and **20%** for **testing**.
- run the experiment with the end-to-end model composed by **C3D and two fully connected layers**;
- run the experiment with the end-to-end model based on the **ConvLSTM** architecture (with two fully connected layers for classification);

### 5.1 C3D (until "fc6") + Fully Connected Layers

The following table shows the layers of the end to end model composed of C3D (until the first fully connected layer - fc6) and two fully-connected layers for classification. Note that, in the experiments, **only the last two dense layer are trained** (the weights of C3D are not modified), in a **transfer learning** manner.

| Layer Type                                     | Output Shape             | Parameter # |
|:-----------------------------------------------|:-------------------------|------------:|
| Conv3D, *3x3x3*, *stride=1*                    | (None, 16, 112, 112, 64) |        5248 |
| MaxPooling3D, *1x2x2*                          | (None, 16, 56, 56, 64)   |           0 |
| Conv3D, *3x3x3*, *stride=1*                    | (None, 16, 56, 56, 128)  |      221312 |
| MaxPooling3D, *2x2x2*                          | (None, 8, 28, 28, 128)   |           0 |
| Conv3D, *3x3x3*, *stride=1*                    | (None, 8, 28, 28, 256)   |      884992 |
| Conv3D, *3x3x3*, *stride=1*                    | (None, 8, 28, 28, 256)   |     1769728 |
| MaxPooling3D, *2x2x2*                          | (None, 4, 14, 14, 256)   |           0 |
| Conv3D, *3x3x3*, *stride=1*                    | (None, 4, 14, 14, 512)   |     3539456 |
| Conv3D, *3x3x3*, *stride=1*                    | (None, 4, 14, 14, 512)   |     7078400 |
| MaxPooling3D, *2x2x2*                          | (None, 2, 7, 7, 512)     |           0 |
| Conv3D, *3x3x3*, *stride=1*                    | (None, 2, 7, 7, 512)     |     7078400 |
| Conv3D, *3x3x3*, *stride=1*                    | (None, 2, 7, 7, 512)     |     7078400 |
| ZeroPadding3D                                  | (None, 2, 8, 8, 512)     |           0 |
| MaxPooling3D, *2x2x2*                          | (None, 1, 4, 4, 512)     |           0 |
| Flatten                                        | (None, 8192)             |           0 |
| Dense, *4096 units*, *ReLU activation*         | (None, 4096)             |    33558528 |
| Dropout, *0.5*                                 | (None, 4096)             |           0 |
| Dense, *512 units*, *ReLU activation*          | (None, 512)              |     2097664 |
| Dropout, *0.5*                                 | (None, 512)              |           0 |
| Dense,  *1 unit*, *Sigmoid activation*         | (None, 1)                |         513 |


### 5.2 End-to-End ConvLSTM
The following table shows the layers of the end-to-end model based on the ConvLSTM architecture. Note that **the entire network is trained**.

| Layer Type                                     | Output Shape         | Parameter # |
|:-----------------------------------------------|:---------------------|------------:|
| ConvLSTM2D, *64 3x3 filters*                   | (None, 110, 110, 64) |      154624 |
| Dropout, *0.5*                                 | (None, 110, 110, 64) |           0 |
| Flatten                                        | (None, 774400)       |           0 |
| Dense, *256 units*, *ReLU activation*          | (None, 256)          |   198246656 |
| Dropout, *0.5*                                 | (None, 256)          |           0 |
| Dense,  *1 unit*, *Sigmoid activation*         | (None, 1)            |         267 |


In [6]:
from cv2.gapi import video
# definitions of two end-to-end models + definitions of experiments
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit, StratifiedGroupKFold
from sklearn.metrics import roc_curve, auc, accuracy_score, confusion_matrix, classification_report
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pylab as plt
import os
# from keras.models import Sequential, Model
# from keras.layers import Input, Dense, Dropout, Flatten, ConvLSTM2D

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv3D, MaxPooling3D, ZeroPadding3D
from tensorflow.keras.layers import Dense, Dropout, Flatten, ConvLSTM2D


def getC3DCNNModel(verbose=True):
    """Creates the C3D + fully connected layers end-to-end model object with the
    sequential API: https://keras.io/models/sequential/

    Parameters
    ----------
    verbose : bool
              if True prints the model summary (default True)

    Returns
    -------
    model : Sequential
            The instantiated model
    """
    pretrainedModel = getFeatureExtractor('weights/C3D_Sport1M_weights_keras_2.2.4.h5', 'fc6', False)
    for layer in pretrainedModel.layers:
        layer.trainable = False
        # layer.trainable = True

    dropout1 = Dropout(.5)(pretrainedModel.output)
    fc7Alt = Dense(512, activation='relu', name='fc7-alt')(dropout1)
    dropout2 = Dropout(.5)(fc7Alt)
    if is_binary == True:
      output = Dense(1, activation='sigmoid')(dropout2)
    else:
      output = Dense(3, activation='softmax')(dropout2)
    model = Model(inputs=pretrainedModel.inputs, outputs=output)
    if verbose:
        model.summary()
    if is_binary == True:
      model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    else:
      model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    del pretrainedModel

    return model

def getLSTMModel(verbose=True):
    """Creates the ConvLSTM + fully connected layers end-to-end model object
    with the sequential API: https://keras.io/models/sequential/

    Parameters
    ----------
    verbose : bool
              if True prints the model summary (default True)

    Returns
    -------
    model : Sequential
            The instantiated model
    """
    model = Sequential()
    model.add(ConvLSTM2D(filters=64, kernel_size=(3,3), input_shape=(16,112,112,3)))
    model.add(Dropout(0.5))
    model.add(Flatten())
    model.add(Dense(256, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(1, activation='sigmoid'))
    if verbose:
        model.summary()
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

    return model

# def runEndToEndExperiment(getModel, batchSize, datasetBasePath, mmapDatasetBasePath, samplesMMapName, lablesMMapName, endToEndModelName, rState):
#     """"Runs the tests with end to end models.

#     Parameters
#     ----------
#     getModel : Callable[[bool], Sequential]
#                Function that instantiates the model to be tested
#     batchSize : int
#                 Batch size to be used for training and testing
#     datasetBasePath : str
#                       Pathname to the base of the feature files repository,
#                       which contains two directories, violent and non-violent,
#                       which are divided into cam1 and cam2.
#     mmapDatasetBasePath : str
#                           Folder including the memory maps of samples and labels.
#     samplesMMapName : str
#                       Name of the file storing the numpy array of the samples,
#                       with shape (Sample #, 16, 112, 112, 3). For the AIRTLab
#                       dataset the sample number is 3537.
#     lablesMMapName : str
#                      Name of the file storing the numpy array of the labels,
#                      with shape (Sample #,). For the AIRTLab dataset the sample
#                      number is 3537.
#     endToEndModelName : str
#                         Model name to be used in the AUC-ROC plot.
#     rState : int, RandomState instance or None
#              Controls the randomness of the training and testing indices produced.
#              Pass an int for reproducible output across multiple function calls.

#     """
#     chunk_number = count_chunks(datasetBasePath)
#     X = np.memmap(os.path.join(mmapDatasetBasePath, samplesMMapName), mode='r', dtype=np.float32, shape=(chunk_number, 16, 112, 112, 3))
#     y = np.memmap(os.path.join(mmapDatasetBasePath, lablesMMapName), mode='r', dtype=np.int8, shape=(chunk_number))

#     nsplits = 5
#     #cv = StratifiedKFold(n_splits=nsplits, shuffle=True)
#     cv = StratifiedShuffleSplit(n_splits=nsplits, train_size=0.8, random_state = rState)

#     tprs = []
#     aucs = []
#     scores = []
#     sens = np.zeros(shape=(nsplits))
#     specs = np.zeros(shape=(nsplits))
#     f1Scores = np.zeros(shape=(nsplits))
#     mean_fpr = np.linspace(0, 1, 100)
#     plt.figure(num=1, figsize=(10,10))
#     i = 1

#     for train, test in cv.split(X, y):

#         # train = sklearn.utils.shuffle(train)

#         X_train = np.memmap(os.path.join(mmapDatasetBasePath, 'samples_train.mmap'), mode='w+', dtype=np.float32, shape=X[train].shape)
#         X_train[:] = X[train][:]

#         X_test = np.memmap(os.path.join(mmapDatasetBasePath, 'samples_test.mmap'), mode='w+', dtype=np.float32, shape=X[test].shape)
#         X_test[:] = X[test][:]

#         del X

#         input_shape = (16, 112, 112, 3)
#         if is_binary == True:
#           num_classes = 1
#         else:
#           num_classes = 3
#         conv_filters=64
#         lstm_units=64
#         # model = getModel(input_shape, num_classes, conv_filters, lstm_units)
#         model = getModel(i==1)

#         es = EarlyStopping(monitor='val_loss', mode='min', patience=10, verbose=1, restore_best_weights=True)
#         model.fit(X_train, y[train], validation_split=0.125, epochs=100, batch_size=batchSize, verbose=1, callbacks=[es])

#         del X_train

#         print("Computing scores...")
#         evaluation = model.evaluate(X_test, y[test])
#         scores.append(evaluation)
#         print("Computing probs...")
#         probas = model.predict(X_test, batch_size=batchSize, verbose=1).ravel()
#         del X_test

#         # Compute ROC curve and area the curve
#         fpr, tpr, thresholds = roc_curve(y[test], probas)
#         tprs.append(np.interp(mean_fpr, fpr, tpr))
#         roc_auc = auc(fpr, tpr)
#         aucs.append(roc_auc)
#         plt.plot(fpr, tpr, lw=2, alpha=0.3, label='ROC split %d (AUC = %0.4f)' % (i, roc_auc))

#         y_pred = np.round(probas)
#         report = classification_report(y[test], y_pred, target_names=['non-violent', 'violent'], output_dict=True)
#         sens[i - 1] = report['violent']['recall']
#         specs[i - 1] = report['non-violent']['recall']
#         f1Scores[i - 1] = report['violent']['f1-score']

#         print('confusion matrix split ' + str(i))
#         print(confusion_matrix(y[test], y_pred))
#         print(classification_report(y[test], y_pred, target_names=['non-violent', 'violent']))
#         print('Loss: ' + str(evaluation[0]))
#         print('Accuracy: ' + str(evaluation[1]))
#         print('\n')

#         i += 1

#         X = np.memmap(os.path.join(mmapDatasetBasePath, samplesMMapName), mode='r', dtype=np.float32, shape=(chunk_number, 16, 112, 112, 3))
#         del report
#         del model

#     plt.plot([0, 1], [0, 1], linestyle='--', lw=2, color='r', label='Chance', alpha=.8)

#     mean_tpr = np.mean(tprs, axis=0)
#     mean_auc = auc(mean_fpr, mean_tpr)
#     std_auc = np.std(aucs)
#     plt.plot(mean_fpr, mean_tpr, color='b', label=r'Mean ROC (AUC = %0.4f $\pm$ %0.4f)' % (mean_auc, std_auc), lw=2, alpha=.8)

#     std_tpr = np.std(tprs, axis=0)
#     tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
#     tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
#     plt.fill_between(mean_fpr, tprs_lower, tprs_upper, color='grey', alpha=.2, label=r'$\pm$ 1 std. dev.')

#     plt.xlim([-0.01, 1.01])
#     plt.ylim([-0.01, 1.01])
#     plt.xlabel('False Positive Rate',fontsize=18)
#     plt.ylabel('True Positive Rate',fontsize=18)
#     plt.title('Cross-Validation ROC of ' + endToEndModelName  + ' model',fontsize=18)
#     plt.legend(loc="lower right", prop={'size': 15})

#     plt.savefig(endToEndModelName.replace('+', '') + '.pdf')
#     plt.show()

#     #print(scores)
#     np_scores = np.array(scores)
#     losses = np_scores[:, 0:1]
#     accuracies = np_scores[:, 1:2]
#     print('Losses')
#     print(losses)
#     print('Accuracies')
#     print(accuracies)
#     print('Sensitivities')
#     print(sens)
#     print('specificities')
#     print(specs)
#     print('F1-scores')
#     print(f1Scores)
#     print("Avg loss: {0} +/- {1}".format(np.mean(losses), np.std(losses)))
#     print("Avg accuracy: {0} +/- {1}".format(np.mean(accuracies), np.std(accuracies)))
#     print("Avg sensitivity: {0} +/- {1}".format(np.mean(sens), np.std(sens)))
#     print("Avg specificity: {0} +/- {1}".format(np.mean(specs), np.std(specs)))
#     print("Avg f1-score: {0} +/- {1}".format(np.mean(f1Scores), np.std(f1Scores)))

#     del X
#     del y
#     del sens
#     del specs
#     del f1Scores
#     del accuracies
#     del losses
#     del np_scores



def runEndToEndExperiment(getModel, batchSize, datasetBasePath, mmapDatasetBasePath,
                          samplesMMapName, lablesMMapName, endToEndModelName, rState, class_names=None):
    """"Runs the tests with end to end models.

    Notes for multiclass (is_binary=False):
      - Expects labels y in {0,1,2} (int)
      - Expects model.predict(X) to return probabilities with shape (N, 3)
      - Uses argmax for predictions
      - Reports macro-avg precision/recall/F1 + accuracy + confusion matrices
      - (Optional) computes macro ROC-AUC (OvR) numerically (no ROC plot)

    Parameters
    ----------
    getModel : Callable[[bool], keras.Model]
               Function that instantiates the model to be tested.
               Called as getModel(i==1). Ensure it builds binary or 3-class model
               consistently with `is_binary`.
    batchSize : int
    datasetBasePath : str
    mmapDatasetBasePath : str
    samplesMMapName : str
    lablesMMapName : str
    endToEndModelName : str
    rState : int or None
    is_binary : bool
    class_names : list[str] or None
                  For multiclass, supply something like ["class0","class1","class2"].
    """
    import os
    import numpy as np
    import matplotlib.pyplot as plt

    from sklearn.model_selection import StratifiedShuffleSplit
    from sklearn.metrics import (roc_curve, auc, confusion_matrix,
                                 classification_report, roc_auc_score)
    from sklearn.preprocessing import label_binarize

    chunk_number = count_chunks(datasetBasePath)

    X = np.memmap(os.path.join(mmapDatasetBasePath, samplesMMapName),
                  mode='r', dtype=np.float32,
                  shape=(chunk_number, 16, 112, 112, 3))
    y = np.memmap(os.path.join(mmapDatasetBasePath, lablesMMapName),
                  mode='r', dtype=np.int8,
                  shape=(chunk_number,))
    videoname = np.memmap(os.path.join(mmapDatasetBasePath, 'videos.mmap'),
                          mode='r', dtype=np.int8,
                          shape=(chunk_number,))



    nsplits = 5
    # cv = StratifiedShuffleSplit(n_splits=nsplits, train_size=0.8, random_state=rState)
    cv = StratifiedGroupKFold(
        n_splits=nsplits,
        shuffle=True,
        random_state=rState
    )

    scores = []

    # ----- metrics containers -----
    if is_binary:
        tprs = []
        aucs = []
        sens = np.zeros(shape=(nsplits))
        specs = np.zeros(shape=(nsplits))
        f1Scores = np.zeros(shape=(nsplits))
        mean_fpr = np.linspace(0, 1, 100)
        plt.figure(num=1, figsize=(10, 10))
    else:
        # multiclass: macro metrics
        macro_precisions = np.zeros(shape=(nsplits))
        macro_recalls = np.zeros(shape=(nsplits))
        macro_f1s = np.zeros(shape=(nsplits))
        macro_aucs = np.zeros(shape=(nsplits))  # numeric macro AUC (OvR)
        if class_names is None:
            class_names = ["0", "1", "2"]

    i = 1


    # for train, test in cv.split(X, y):
    for train, test in cv.split(X, y, groups=videoname):


        # Build train/test memmaps for this split
        X_train = np.memmap(os.path.join(mmapDatasetBasePath, 'samples_train.mmap'),
                            mode='w+', dtype=np.float32, shape=X[train].shape)
        X_train[:] = X[train][:]

        X_test = np.memmap(os.path.join(mmapDatasetBasePath, 'samples_test.mmap'),
                           mode='w+', dtype=np.float32, shape=X[test].shape)
        X_test[:] = X[test][:]


        perm = np.random.RandomState(rState).permutation(len(train))
        X_train = X_train[perm]


        # Flush to disk (safer for memmap)
        # X_train.flush()
        # X_test.flush()

        # Free main X mapping to reduce memory pressure; remap later
        del X

        #         input_shape = (16, 112, 112, 3)
        #         if is_binary == True:
        #           num_classes = 1
        #         else:
        #           num_classes = 3
        #         conv_filters=64
        #         lstm_units=64
        #         # model = getModel(input_shape, num_classes, conv_filters, lstm_units)


        # Build model
        model = getModel(i == 1)  # Ensure your getModel aligns with `is_binary`

        es = EarlyStopping(monitor='val_loss', mode='min', patience=10,
                           verbose=1, restore_best_weights=True)

        model.fit(X_train, np.asarray(y)[train][perm],
                  validation_split=0.125,
                  epochs=120,
                  batch_size=batchSize,
                  verbose=1,
                  callbacks=[es])

        # release train memmap
        del X_train

        print("Computing scores...")
        evaluation = model.evaluate(X_test, y[test], verbose=1)
        scores.append(evaluation)

        print("Computing probs...")
        probas = model.predict(X_test, batch_size=batchSize, verbose=1)

        # release test memmap ASAP after we have predictions (optional)
        del X_test

        # ------------------------------------------------------------
        #                     BINARY vs MULTICLASS
        # ------------------------------------------------------------
        if is_binary == True:
            # binary code here
            probas_1d = np.asarray(probas).ravel()
            y_pred = (probas_1d >= 0.5).astype(int)

            # ROC curve and AUC for this split
            fpr, tpr, thresholds = roc_curve(y[test], probas_1d)
            tprs.append(np.interp(mean_fpr, fpr, tpr))
            roc_auc = auc(fpr, tpr)
            aucs.append(roc_auc)
            plt.plot(fpr, tpr, lw=2, alpha=0.3,
                     label='ROC split %d (AUC = %0.4f)' % (i, roc_auc))

            report = classification_report(
                y[test], y_pred,
                target_names=['non-violent', 'violent'],
                output_dict=True
            )
            sens[i - 1] = report['violent']['recall']
            specs[i - 1] = report['non-violent']['recall']
            f1Scores[i - 1] = report['violent']['f1-score']

            print('confusion matrix split ' + str(i))
            print(confusion_matrix(y[test], y_pred))
            print(classification_report(y[test], y_pred,
                                        target_names=['non-violent', 'violent']))
            print('Loss: ' + str(evaluation[0]))
            print('Accuracy: ' + str(evaluation[1]))
            print('\n')

            del report

        else:
            # 3 classes classification code here
            probas_2d = np.asarray(probas)  # expected (N,3)
            y_pred = np.argmax(probas_2d, axis=1)

            report = classification_report(
                y[test], y_pred,
                target_names=class_names,
                output_dict=True
            )

            macro_precisions[i - 1] = report['macro avg']['precision']
            macro_recalls[i - 1] = report['macro avg']['recall']
            macro_f1s[i - 1] = report['macro avg']['f1-score']

            # numeric macro ROC-AUC (OvR) if possible
            try:
                y_true_bin = label_binarize(y[test], classes=list(range(len(class_names))))
                macro_aucs[i - 1] = roc_auc_score(
                    y_true_bin, probas_2d,
                    average="macro",
                    multi_class="ovr"
                )
            except Exception:
                macro_aucs[i - 1] = np.nan

            print('confusion matrix split ' + str(i))
            print(confusion_matrix(y[test], y_pred))
            print(classification_report(y[test], y_pred, target_names=class_names))
            print('Loss: ' + str(evaluation[0]))
            print('Accuracy: ' + str(evaluation[1]))
            print('Macro Precision: ' + str(macro_precisions[i - 1]))
            print('Macro Recall: ' + str(macro_recalls[i - 1]))
            print('Macro F1: ' + str(macro_f1s[i - 1]))
            print('Macro ROC-AUC (OvR): ' + str(macro_aucs[i - 1]))
            print('\n')

            del report

        i += 1

        # Remap X for next split
        X = np.memmap(os.path.join(mmapDatasetBasePath, samplesMMapName),
                      mode='r', dtype=np.float32,
                      shape=(chunk_number, 16, 112, 112, 3))

        del model
        del probas

    # ------------------------------------------------------------
    #                        SUMMARY / PLOTS
    # ------------------------------------------------------------
    if is_binary == True:
        # binary ROC plot summary
        plt.plot([0, 1], [0, 1], linestyle='--', lw=2, color='r',
                 label='Chance', alpha=.8)

        mean_tpr = np.mean(tprs, axis=0)
        mean_auc = auc(mean_fpr, mean_tpr)
        std_auc = np.std(aucs)
        plt.plot(mean_fpr, mean_tpr, color='b',
                 label=r'Mean ROC (AUC = %0.4f $\pm$ %0.4f)' % (mean_auc, std_auc),
                 lw=2, alpha=.8)

        std_tpr = np.std(tprs, axis=0)
        tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
        tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
        plt.fill_between(mean_fpr, tprs_lower, tprs_upper,
                         color='grey', alpha=.2, label=r'$\pm$ 1 std. dev.')

        plt.xlim([-0.01, 1.01])
        plt.ylim([-0.01, 1.01])
        plt.xlabel('False Positive Rate', fontsize=18)
        plt.ylabel('True Positive Rate', fontsize=18)
        plt.title('Cross-Validation ROC of ' + endToEndModelName + ' model', fontsize=18)
        plt.legend(loc="lower right", prop={'size': 15})

        plt.savefig(endToEndModelName.replace('+', '') + '.pdf')
        plt.show()

    else:
        # multiclass: no ROC plot (by default)
        print("Multiclass run finished (no ROC curve plotted).")

    # scores summary
    np_scores = np.array(scores, dtype=float)
    losses = np_scores[:, 0:1]
    accuracies = np_scores[:, 1:2]

    print('Losses')
    print(losses)
    print('Accuracies')
    print(accuracies)

    if is_binary == True:
        print('Sensitivities')
        print(sens)
        print('specificities')
        print(specs)
        print('F1-scores')
        print(f1Scores)

        print("Avg loss: {0} +/- {1}".format(np.mean(losses), np.std(losses)))
        print("Avg accuracy: {0} +/- {1}".format(np.mean(accuracies), np.std(accuracies)))
        print("Avg sensitivity: {0} +/- {1}".format(np.mean(sens), np.std(sens)))
        print("Avg specificity: {0} +/- {1}".format(np.mean(specs), np.std(specs)))
        print("Avg f1-score: {0} +/- {1}".format(np.mean(f1Scores), np.std(f1Scores)))

        del sens, specs, f1Scores, tprs, aucs, mean_fpr

    else:
        print('Macro Precisions')
        print(macro_precisions)
        print('Macro Recalls')
        print(macro_recalls)
        print('Macro F1-scores')
        print(macro_f1s)
        print('Macro ROC-AUC (OvR)')
        print(macro_aucs)

        print("Avg loss: {0} +/- {1}".format(np.mean(losses), np.std(losses)))
        print("Avg accuracy: {0} +/- {1}".format(np.mean(accuracies), np.std(accuracies)))
        print("Avg macro precision: {0} +/- {1}".format(np.mean(macro_precisions), np.std(macro_precisions)))
        print("Avg macro recall: {0} +/- {1}".format(np.mean(macro_recalls), np.std(macro_recalls)))
        print("Avg macro f1: {0} +/- {1}".format(np.mean(macro_f1s), np.std(macro_f1s)))
        # macro_aucs may contain nan if computation failed
        print("Avg macro auc: {0} +/- {1}".format(np.nanmean(macro_aucs), np.nanstd(macro_aucs)))

        del macro_precisions, macro_recalls, macro_f1s, macro_aucs

    del X, y, accuracies, losses, np_scores


In [7]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv3D, MaxPooling3D, Flatten, Dense, LSTM, TimeDistributed, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K

def get3DCNNLSTMModel(verbose=True):
    """Creates the 3DCNN + LSTM

    Parameters
    ----------
    verbose : bool
              if True prints the model summary (default True)

    Returns
    -------
    model : Sequential
            The instantiated model
    """
    pretrainedModel = getFeatureExtractor('weights/C3D_Sport1M_weights_keras_2.2.4.h5', 'pool3', False)
    for layer in pretrainedModel.layers:
        layer.trainable = False
        # layer.trainable = True

    # (batch, 10, conv_filters)
    x = TimeDistributed(Flatten())(pretrainedModel.output)
    # (batch, lstm_units)
    x = LSTM(64)(x)
    x = Dropout(.5)(x)
    # (batch, 256)
    x = Dense(256, activation='relu')(x)

    if is_binary == True:
      outputs = Dense(1, activation='sigmoid')(x)
    else:
      outputs = Dense(3, activation='softmax')(x)

    model = Model(pretrainedModel.inputs, outputs)
    if verbose:
        model.summary()

    # binary
    # learning_rate=0.000001
    # Accuracies
    # [[0.95480227]
    # [0.92231637]
    # [0.93502825]
    # [0.93644071]
    # [0.93926555]]

    # learning_rate=0.000005
    # Accuracies
    # [[0.96327686]
    # [0.95480227]
    # [0.95762712]
    # [0.96045196]
    # [0.970339  ]]

    # learning_rate=0.00001
    # Accuracies
    # [[0.96610171]
    # [0.95338982]
    # [0.94350284]
    # [0.94915253]
    # [0.95197737]]

    # 3 classes
    # learning_rate=0.000001
    # Accuracies
    # [[0.80225986]
    # [0.81779659]
    # [0.83757061]
    # [0.81073445]
    # [0.83615822]]

    # learning_rate=0.000005
    # Accuracies
    # [[0.89548022]
    # [0.87005651]
    # [0.8742938 ]
    # [0.8757062 ]
    # [0.86864406]]

    # learning_rate=0.00001
    # Accuracies
    # [[0.87005651]
    # [0.85310733]
    # [0.87711865]
    # [0.83333331]
    # [0.84463274]]


    if is_binary == True:
      model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.000005, beta_1=0.9, beta_2=0.999, epsilon=1e-08),
                    loss='binary_crossentropy', metrics=['accuracy'])
    else:
      model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.000005, beta_1=0.9, beta_2=0.999, epsilon=1e-08),
                    loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    # model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    del pretrainedModel

    # dropout1 = Dropout(.5)(pretrainedModel.output)
    # fc7Alt = Dense(512, activation='relu', name='fc-alt')(dropout1)
    # dropout2 = Dropout(.5)(fc7Alt)
    # output = Dense(1, activation='sigmoid')(dropout2)
    # model = Model(inputs=pretrainedModel.inputs, outputs=output)
    # if verbose:
    #     model.summary()
    # model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

    # del pretrainedModel

    return model

# change maxpooling3D from (2,2,2) to (1,2,2). This old code, may delete
def create_model(input_shape, num_classes, conv_filters, lstm_units, dropout_rate=0.5):
    # (batch, 10, 64, 64, 3)
    inputs = Input(shape=input_shape)
    # (batch, 10, 64, 64, conv_filters)
    x = Conv3D(conv_filters, (3, 3, 3), activation='relu', padding='same')(inputs)
    # (batch, 10, 32, 32, conv_filters)
    x = MaxPooling3D((1, 2, 2))(x)

    # (batch, 10, 32, 32, conv_filters)
    x = Conv3D(conv_filters, (3, 3, 3), activation='relu', padding='same')(x)
    # (batch, 10, 16, 16, conv_filters)
    x = MaxPooling3D((1, 2, 2))(x)

    # (batch, 10, 16, 16, conv_filters)
    x = Conv3D(conv_filters, (3, 3, 3), activation='relu', padding='same')(x)
    # (batch, 10, 8, 8, conv_filters)
    x = MaxPooling3D((1, 2, 2))(x)

    # (batch, 10, conv_filters)
    x = TimeDistributed(Flatten())(x)
    # (batch, lstm_units)
    x = LSTM(lstm_units)(x)
    x = Dropout(dropout_rate)(x)
    # (batch, 256)
    x = Dense(256, activation='relu')(x)
    outputs = Dense(num_classes, activation='sigmoid')(x)

    model = Model(inputs, outputs)
    # model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    print(model.summary())

    return model


input_shape = (16, 112, 112, 3)
num_classes = 3
# conv_filters_list = [32, 64, 128]
# lstm_units_list = [64, 128]
conv_filters_list = [64]
lstm_units_list = [64]
results_neurons = {}

# early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
mean_fpr = np.linspace(0, 1, 100)

In [8]:
# Experiment 0: 3dcnnlstm
# ❌ 1. Temporal pooling too aggressive
# You destroy temporal structure before the LSTM.

# ❌ 2. LSTM is a severe bottleneck
# 4096 (C3D FC6)  vs  64 (your LSTM)
# This is a 64× compression with no justification.

# ❌ 3. LSTM receives flattened spatial tensors
# This mixes space + time in a way that hurts optimization.

# ❌ 4. No pretrained temporal inductive bias
# C3D has one; your CNN+LSTM does not.
# runEndToEndExperiment(get3DCNNLSTMModel, 8, '/datarepo/violence-detection-dataset',
#                       'airtlabDataset', 'samples.mmap', 'labels.mmap', '3dcnnlstm', 42)
runEndToEndExperiment(get3DCNNLSTMModel, 8, BASE_DIR/'data/processed/violence-detection-dataset',
                      'airtlabDataset', 'samples.mmap', 'labels.mmap', '3dcnnlstm', 42)

I0000 00:00:1771544968.510066   53832 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1771544969.306677   53832 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1771544969.306801   53832 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1771544969.350471   53832 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1771544969.350625   53832 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:0

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 c3d_input (InputLayer)      [(None, 16, 112, 112, 3   0         
                             )]                                  
                                                                 
 conv1 (Conv3D)              (None, 16, 112, 112, 64   5248      
                             )                                   
                                                                 
 pool1 (MaxPooling3D)        (None, 16, 56, 56, 64)    0         
                                                                 
 conv2 (Conv3D)              (None, 16, 56, 56, 128)   221312    
                                                                 
 pool2 (MaxPooling3D)        (None, 8, 28, 28, 128)    0         
                                                                 
 conv3a (Conv3D)             (None, 8, 28, 28, 256)    884992

2026-02-19 23:50:09.288923: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 90701
W0000 00:00:1771545009.507452   61844 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1771545009.582587   61844 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1771545009.612409   61844 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1771545009.663985   61844 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1771545009.679482   61844 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1771545009.692479   61844 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1771545009.705570   61844 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1771545009.855532   61844 gpu_

InternalError: Graph execution error:

Detected at node CudnnRNN defined at (most recent call last):
<stack traces unavailable>
Failed to call DoRnnForward with model config: [rnn_mode, rnn_input_mode, rnn_direction_mode]: 2, 0, 0 , [num_layers, input_size, num_units, dir_count, max_seq_length, batch_size, cell_num_units]: [1, 50176, 64, 1, 4, 8, 64] 
	 [[{{node CudnnRNN}}]]
	 [[model/lstm/PartitionedCall]] [Op:__inference_train_function_3552]

In [ ]:
# Using ensemble method to test on video level

def runEndToEndExperiment(getModel, batchSize, datasetBasePath, mmapDatasetBasePath,
                          samplesMMapName, lablesMMapName, endToEndModelName, rState, class_names=None):
    """"Runs the tests with end to end models.

    Notes for multiclass (is_binary=False):
      - Expects labels y in {0,1,2} (int)
      - Expects model.predict(X) to return probabilities with shape (N, 3)
      - Uses argmax for predictions
      - Reports macro-avg precision/recall/F1 + accuracy + confusion matrices
      - (Optional) computes macro ROC-AUC (OvR) numerically (no ROC plot)

    Parameters
    ----------
    getModel : Callable[[bool], keras.Model]
               Function that instantiates the model to be tested.
               Called as getModel(i==1). Ensure it builds binary or 3-class model
               consistently with `is_binary`.
    batchSize : int
    datasetBasePath : str
    mmapDatasetBasePath : str
    samplesMMapName : str
    lablesMMapName : str
    endToEndModelName : str
    rState : int or None
    is_binary : bool
    class_names : list[str] or None
                  For multiclass, supply something like ["class0","class1","class2"].
    """
    import os
    import numpy as np
    import matplotlib.pyplot as plt

    from sklearn.model_selection import StratifiedShuffleSplit
    from sklearn.metrics import (roc_curve, auc, confusion_matrix,
                                 classification_report, roc_auc_score)
    from sklearn.preprocessing import label_binarize

    # =========================
    # [CHANGED/ADDED] ENSEMBLE SETTINGS + HELPER
    # =========================
    ensemble_threshold = 0.34  # chunk-vote ratio must be > 0.5 to "accept" a video prediction

    def _video_ensemble_majority_vote(y_true_chunks, y_pred_chunks, video_ids, threshold, n_classes):
        """
        For each video:
          - video_true = majority label in y_true_chunks for that video (safe)
          - video_pred = majority label in y_pred_chunks for that video
          - support = max(pred_count)/num_chunks
          - video is counted as "classified" only if support > threshold
        Returns:
          video_acc_on_covered, coverage, (optional lists)
        """
        y_true_chunks = np.asarray(y_true_chunks).astype(int)
        y_pred_chunks = np.asarray(y_pred_chunks).astype(int)
        video_ids = np.asarray(video_ids)

        unique_videos = np.unique(video_ids)

        covered = 0
        correct = 0

        for vid in unique_videos:
            idx = (video_ids == vid)

            yt = y_true_chunks[idx]
            yp = y_pred_chunks[idx]

            # true label (use majority in case of noise)
            true_counts = np.bincount(yt, minlength=n_classes)
            video_true = int(np.argmax(true_counts))

            # predicted label by majority vote across chunks
            pred_counts = np.bincount(yp, minlength=n_classes)
            video_pred = int(np.argmax(pred_counts))

            support = float(np.max(pred_counts)) / float(len(yp))  # fraction of chunks voting for the top class

            # accept prediction only if confident enough
            if support > threshold:
                covered += 1
                if video_pred == video_true:
                    correct += 1

        coverage = covered / float(len(unique_videos)) if len(unique_videos) > 0 else 0.0
        video_acc_on_covered = correct / float(covered) if covered > 0 else 0.0
        return video_acc_on_covered, coverage
    # =========================

    chunk_number = count_chunks(datasetBasePath)

    X = np.memmap(os.path.join(mmapDatasetBasePath, samplesMMapName),
                  mode='r', dtype=np.float32,
                  shape=(chunk_number, 16, 112, 112, 3))
    y = np.memmap(os.path.join(mmapDatasetBasePath, lablesMMapName),
                  mode='r', dtype=np.int8,
                  shape=(chunk_number,))
    videoname = np.memmap(os.path.join(mmapDatasetBasePath, 'videos.mmap'),
                          mode='r', dtype=np.int8,
                          shape=(chunk_number,))



    nsplits = 5
    # cv = StratifiedShuffleSplit(n_splits=nsplits, train_size=0.8, random_state=rState)
    cv = StratifiedGroupKFold(
        n_splits=nsplits,
        shuffle=True,
        random_state=rState
    )

    scores = []

    # ----- metrics containers -----
    if is_binary:
        tprs = []
        aucs = []
        sens = np.zeros(shape=(nsplits))
        specs = np.zeros(shape=(nsplits))
        f1Scores = np.zeros(shape=(nsplits))
        mean_fpr = np.linspace(0, 1, 100)
        plt.figure(num=1, figsize=(10, 10))
    else:
        # multiclass: macro metrics
        macro_precisions = np.zeros(shape=(nsplits))
        macro_recalls = np.zeros(shape=(nsplits))
        macro_f1s = np.zeros(shape=(nsplits))
        macro_aucs = np.zeros(shape=(nsplits))  # numeric macro AUC (OvR)
        if class_names is None:
            class_names = ["0", "1", "2"]

    # =========================
    # [ADDED] ENSEMBLE METRIC CONTAINERS (for both binary and 3-class)
    # =========================
    video_accs = np.zeros(shape=(nsplits))
    video_covs = np.zeros(shape=(nsplits))
    # =========================

    i = 1


    # for train, test in cv.split(X, y):
    for train, test in cv.split(X, y, groups=videoname):


        # Build train/test memmaps for this split
        X_train = np.memmap(os.path.join(mmapDatasetBasePath, 'samples_train.mmap'),
                            mode='w+', dtype=np.float32, shape=X[train].shape)
        X_train[:] = X[train][:]

        X_test = np.memmap(os.path.join(mmapDatasetBasePath, 'samples_test.mmap'),
                           mode='w+', dtype=np.float32, shape=X[test].shape)
        X_test[:] = X[test][:]


        perm = np.random.RandomState(rState).permutation(len(train))
        X_train = X_train[perm]


        # Flush to disk (safer for memmap)
        # X_train.flush()
        # X_test.flush()

        # Free main X mapping to reduce memory pressure; remap later
        del X

        #         input_shape = (16, 112, 112, 3)
        #         if is_binary == True:
        #           num_classes = 1
        #         else:
        #           num_classes = 3
        #         conv_filters=64
        #         lstm_units=64
        #         # model = getModel(input_shape, num_classes, conv_filters, lstm_units)


        # Build model
        model = getModel(i == 1)  # Ensure your getModel aligns with `is_binary`

        es = EarlyStopping(monitor='val_loss', mode='min', patience=10,
                           verbose=1, restore_best_weights=True)

        model.fit(X_train, np.asarray(y)[train][perm],
                  validation_split=0.125,
                  epochs=120,
                  batch_size=batchSize,
                  verbose=1,
                  callbacks=[es])

        # release train memmap
        del X_train

        print("Computing scores...")
        evaluation = model.evaluate(X_test, y[test], verbose=1)
        scores.append(evaluation)

        print("Computing probs...")
        probas = model.predict(X_test, batch_size=batchSize, verbose=1)

        # release test memmap ASAP after we have predictions (optional)
        del X_test

        # ------------------------------------------------------------
        #                     BINARY vs MULTICLASS
        # ------------------------------------------------------------
        if is_binary == True:
            # binary code here
            probas_1d = np.asarray(probas).ravel()
            y_pred = (probas_1d >= 0.5).astype(int)

            # =========================
            # [ADDED] ENSEMBLE TESTING (BINARY) AT VIDEO LEVEL
            # =========================
            # majority vote across chunks per video; accept only if support > threshold
            v_acc, v_cov = _video_ensemble_majority_vote(
                y_true_chunks=np.asarray(y)[test],
                y_pred_chunks=y_pred,
                video_ids=np.asarray(videoname)[test],
                threshold=ensemble_threshold,
                n_classes=2
            )
            video_accs[i - 1] = v_acc
            video_covs[i - 1] = v_cov
            print(f"[Ensemble@Video] split {i} | threshold>{ensemble_threshold:.2f} | coverage={v_cov:.4f} | video-acc={v_acc:.4f}")
            # =========================

            # ROC curve and AUC for this split
            fpr, tpr, thresholds = roc_curve(y[test], probas_1d)
            tprs.append(np.interp(mean_fpr, fpr, tpr))
            roc_auc = auc(fpr, tpr)
            aucs.append(roc_auc)
            plt.plot(fpr, tpr, lw=2, alpha=0.3,
                     label='ROC split %d (AUC = %0.4f)' % (i, roc_auc))

            report = classification_report(
                y[test], y_pred,
                target_names=['non-violent', 'violent'],
                output_dict=True
            )
            sens[i - 1] = report['violent']['recall']
            specs[i - 1] = report['non-violent']['recall']
            f1Scores[i - 1] = report['violent']['f1-score']

            print('confusion matrix split ' + str(i))
            print(confusion_matrix(y[test], y_pred))
            print(classification_report(y[test], y_pred,
                                        target_names=['non-violent', 'violent']))
            print('Loss: ' + str(evaluation[0]))
            print('Accuracy: ' + str(evaluation[1]))
            print('\n')

            del report

        else:
            # 3 classes classification code here
            probas_2d = np.asarray(probas)  # expected (N,3)
            y_pred = np.argmax(probas_2d, axis=1)

            # =========================
            # [ADDED] ENSEMBLE TESTING (3-CLASS) AT VIDEO LEVEL
            # =========================
            v_acc, v_cov = _video_ensemble_majority_vote(
                y_true_chunks=np.asarray(y)[test],
                y_pred_chunks=y_pred,
                video_ids=np.asarray(videoname)[test],
                threshold=ensemble_threshold,
                n_classes=len(class_names)
            )
            video_accs[i - 1] = v_acc
            video_covs[i - 1] = v_cov
            print(f"[Ensemble@Video] split {i} | threshold>{ensemble_threshold:.2f} | coverage={v_cov:.4f} | video-acc={v_acc:.4f}")
            # =========================

            report = classification_report(
                y[test], y_pred,
                target_names=class_names,
                output_dict=True
            )

            macro_precisions[i - 1] = report['macro avg']['precision']
            macro_recalls[i - 1] = report['macro avg']['recall']
            macro_f1s[i - 1] = report['macro avg']['f1-score']

            # numeric macro ROC-AUC (OvR) if possible
            try:
                y_true_bin = label_binarize(y[test], classes=list(range(len(class_names))))
                macro_aucs[i - 1] = roc_auc_score(
                    y_true_bin, probas_2d,
                    average="macro",
                    multi_class="ovr"
                )
            except Exception:
                macro_aucs[i - 1] = np.nan

            print('confusion matrix split ' + str(i))
            print(confusion_matrix(y[test], y_pred))
            print(classification_report(y[test], y_pred, target_names=class_names))
            print('Loss: ' + str(evaluation[0]))
            print('Accuracy: ' + str(evaluation[1]))
            print('Macro Precision: ' + str(macro_precisions[i - 1]))
            print('Macro Recall: ' + str(macro_recalls[i - 1]))
            print('Macro F1: ' + str(macro_f1s[i - 1]))
            print('Macro ROC-AUC (OvR): ' + str(macro_aucs[i - 1]))
            print('\n')

            del report

        i += 1

        # Remap X for next split
        X = np.memmap(os.path.join(mmapDatasetBasePath, samplesMMapName),
                      mode='r', dtype=np.float32,
                      shape=(chunk_number, 16, 112, 112, 3))

        del model
        del probas

    # ------------------------------------------------------------
    #                        SUMMARY / PLOTS
    # ------------------------------------------------------------
    if is_binary == True:
        # binary ROC plot summary
        plt.plot([0, 1], [0, 1], linestyle='--', lw=2, color='r',
                 label='Chance', alpha=.8)

        mean_tpr = np.mean(tprs, axis=0)
        mean_auc = auc(mean_fpr, mean_tpr)
        std_auc = np.std(aucs)
        plt.plot(mean_fpr, mean_tpr, color='b',
                 label=r'Mean ROC (AUC = %0.4f $\pm$ %0.4f)' % (mean_auc, std_auc),
                 lw=2, alpha=.8)

        std_tpr = np.std(tprs, axis=0)
        tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
        tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
        plt.fill_between(mean_fpr, tprs_lower, tprs_upper,
                         color='grey', alpha=.2, label=r'$\pm$ 1 std. dev.')

        plt.xlim([-0.01, 1.01])
        plt.ylim([-0.01, 1.01])
        plt.xlabel('False Positive Rate', fontsize=18)
        plt.ylabel('True Positive Rate', fontsize=18)
        plt.title('Cross-Validation ROC of ' + endToEndModelName + ' model', fontsize=18)
        plt.legend(loc="lower right", prop={'size': 15})

        plt.savefig(endToEndModelName.replace('+', '') + '.pdf')
        plt.show()

    else:
        # multiclass: no ROC plot (by default)
        print("Multiclass run finished (no ROC curve plotted).")

    # scores summary
    np_scores = np.array(scores, dtype=float)
    losses = np_scores[:, 0:1]
    accuracies = np_scores[:, 1:2]

    print('Losses')
    print(losses)
    print('Accuracies')
    print(accuracies)

    # =========================
    # [ADDED] ENSEMBLE SUMMARY (BOTH BINARY + 3-CLASS)
    # =========================
    print(f'[Ensemble@Video] threshold>{ensemble_threshold:.2f}')
    print('Video coverage per split')
    print(video_covs)
    print('Video accuracy per split (on covered videos only)')
    print(video_accs)
    print("Avg video coverage: {0} +/- {1}".format(np.mean(video_covs), np.std(video_covs)))
    print("Avg video accuracy: {0} +/- {1}".format(np.mean(video_accs), np.std(video_accs)))
    # =========================

    if is_binary == True:
        print('Sensitivities')
        print(sens)
        print('specificities')
        print(specs)
        print('F1-scores')
        print(f1Scores)

        print("Avg loss: {0} +/- {1}".format(np.mean(losses), np.std(losses)))
        print("Avg accuracy: {0} +/- {1}".format(np.mean(accuracies), np.std(accuracies)))
        print("Avg sensitivity: {0} +/- {1}".format(np.mean(sens), np.std(sens)))
        print("Avg specificity: {0} +/- {1}".format(np.mean(specs), np.std(specs)))
        print("Avg f1-score: {0} +/- {1}".format(np.mean(f1Scores), np.std(f1Scores)))

        del sens, specs, f1Scores, tprs, aucs, mean_fpr

    else:
        print('Macro Precisions')
        print(macro_precisions)
        print('Macro Recalls')
        print(macro_recalls)
        print('Macro F1-scores')
        print(macro_f1s)
        print('Macro ROC-AUC (OvR)')
        print(macro_aucs)

        print("Avg loss: {0} +/- {1}".format(np.mean(losses), np.std(losses)))
        print("Avg accuracy: {0} +/- {1}".format(np.mean(accuracies), np.std(accuracies)))
        print("Avg macro precision: {0} +/- {1}".format(np.mean(macro_precisions), np.std(macro_precisions)))
        print("Avg macro recall: {0} +/- {1}".format(np.mean(macro_recalls), np.std(macro_recalls)))
        print("Avg macro f1: {0} +/- {1}".format(np.mean(macro_f1s), np.std(macro_f1s)))
        # macro_aucs may contain nan if computation failed
        print("Avg macro auc: {0} +/- {1}".format(np.nanmean(macro_aucs), np.nanstd(macro_aucs)))

        del macro_precisions, macro_recalls, macro_f1s, macro_aucs

    # [ADDED] cleanup ensemble arrays
    del video_accs, video_covs

    del X, y, accuracies, losses, np_scores

In [ ]:
# Experiment 2: C3D + fully connected classification
runEndToEndExperiment(getC3DCNNModel, 8, '/datarepo/violence-detection-dataset', 'airtlabDataset', 'samples.mmap', 'labels.mmap', 'C3D + FC', 42)

FileNotFoundError: [Errno 2] No such file or directory: '/datarepo/violence-detection-dataset/non-violence/cam1'

In [ ]:
# Experiment 3: ConvLSTM architecture (trained end-to-end)
runEndToEndExperiment(getLSTMModel, 4, '/datarepo/violence-detection-dataset', 'airtlabDataset', 'samples.mmap', 'labels.mmap', 'ConvLSTM', 42)